In [1]:
# NotY3dGenAI - Advanced 3D Model Generator for Google Colab
# Fixed version - Fully working

import os
import sys
import time
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import plotly.graph_objects as go
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from google.colab import drive, files
import json
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
drive.mount('/content/drive')

# Install required packages
print("📦 Installing required packages...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install -q transformers accelerate diffusers
!pip install -q plotly ipywidgets scikit-image scipy
!pip install -q opencv-python-headless
!pip install -q numpy-stl

# Import after installation
import cv2
from scipy import ndimage
from skimage import measure, filters, morphology
from pathlib import Path

# Fix PIL import issue by patching
import PIL
if not hasattr(PIL, '_typing'):
    class _Ink:
        pass
    PIL._typing = type('_typing', (), {'_Ink': _Ink})()

print("✅ All packages installed successfully!")

# Create directories
Path("/content/noty3d_models").mkdir(exist_ok=True)
Path("/content/drive/MyDrive/NotY3D_Models").mkdir(exist_ok=True)

class NotY3dGenAI:
    def __init__(self):
        self.device = self.setup_device()
        self.setup_models()
        self.create_ui()
        
    def setup_device(self):
        """Setup best available device"""
        if torch.cuda.is_available():
            device = torch.device("cuda")
            print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
            return device
        else:
            print("✅ Using CPU")
            return torch.device("cpu")
    
    def setup_models(self):
        """Initialize models"""
        print("🔄 Setting up generators...")
        self.pipeline = None  # We'll use fallback for reliability
        print("✅ Ready to generate!")
    
    def generate_fallback_image(self, prompt, size=(512, 512)):
        """Generate a sophisticated procedural image from prompt"""
        img = Image.new('RGB', size, color=(20, 20, 40))
        draw = ImageDraw.Draw(img)
        
        # Parse prompt for keywords
        prompt_lower = prompt.lower()
        
        # Determine colors based on prompt
        if any(word in prompt_lower for word in ['man', 'body', 'human', 'person', 'authority']):
            # Human-like form
            center_x, center_y = size[0] // 2, size[1] // 2
            
            # Draw body silhouette
            body_width = size[0] // 3
            body_height = size[1] // 2
            
            # Head
            head_radius = size[0] // 6
            draw.ellipse([center_x - head_radius, center_y - body_height//2 - head_radius,
                         center_x + head_radius, center_y - body_height//2 + head_radius],
                        fill=(200, 180, 150), outline=(100, 80, 60))
            
            # Body
            draw.rectangle([center_x - body_width//2, center_y - body_height//2,
                           center_x + body_width//2, center_y + body_height//2],
                          fill=(150, 130, 110), outline=(100, 80, 60))
            
            # Arms
            arm_width = body_width // 3
            draw.rectangle([center_x - body_width//2 - arm_width, center_y - body_height//4,
                           center_x - body_width//2, center_y + body_height//4],
                          fill=(150, 130, 110), outline=(100, 80, 60))
            draw.rectangle([center_x + body_width//2, center_y - body_height//4,
                           center_x + body_width//2 + arm_width, center_y + body_height//4],
                          fill=(150, 130, 110), outline=(100, 80, 60))
            
            # Legs
            leg_width = body_width // 3
            draw.rectangle([center_x - body_width//3 - leg_width//2, center_y + body_height//2,
                           center_x - body_width//3 + leg_width//2, center_y + body_height//2 + body_height//2],
                          fill=(130, 110, 90), outline=(100, 80, 60))
            draw.rectangle([center_x + body_width//3 - leg_width//2, center_y + body_height//2,
                           center_x + body_width//3 + leg_width//2, center_y + body_height//2 + body_height//2],
                          fill=(130, 110, 90), outline=(100, 80, 60))
            
        elif any(word in prompt_lower for word in ['dragon', 'creature', 'monster', 'mythical']):
            # Dragon-like form
            center_x, center_y = size[0] // 2, size[1] // 2
            
            # Body
            draw.ellipse([center_x - 150, center_y - 100, center_x + 150, center_y + 100],
                        fill=(100, 50, 50), outline=(150, 80, 80))
            
            # Head
            draw.ellipse([center_x + 100, center_y - 60, center_x + 180, center_y + 20],
                        fill=(120, 60, 60), outline=(150, 80, 80))
            
            # Wings
            wing_points = [(center_x - 50, center_y - 80),
                          (center_x - 180, center_y - 200),
                          (center_x - 120, center_y - 100)]
            draw.polygon(wing_points, fill=(80, 40, 40), outline=(150, 80, 80))
            
            wing_points2 = [(center_x + 50, center_y - 80),
                           (center_x + 180, center_y - 200),
                           (center_x + 120, center_y - 100)]
            draw.polygon(wing_points2, fill=(80, 40, 40), outline=(150, 80, 80))
            
        elif any(word in prompt_lower for word in ['car', 'vehicle', 'automobile']):
            # Car-like shape
            center_x, center_y = size[0] // 2, size[1] // 2
            
            # Body
            draw.rectangle([center_x - 180, center_y - 40, center_x + 180, center_y + 40],
                          fill=(50, 100, 150), outline=(30, 70, 110))
            
            # Cabin
            draw.rectangle([center_x - 60, center_y - 80, center_x + 60, center_y - 40],
                          fill=(80, 130, 180), outline=(30, 70, 110))
            
            # Wheels
            draw.ellipse([center_x - 120, center_y + 20, center_x - 80, center_y + 60],
                        fill=(40, 40, 40), outline=(20, 20, 20))
            draw.ellipse([center_x + 80, center_y + 20, center_x + 120, center_y + 60],
                        fill=(40, 40, 40), outline=(20, 20, 20))
        else:
            # Abstract/generic shape
            for i in range(50):
                x = np.random.randint(0, size[0])
                y = np.random.randint(0, size[1])
                r = np.random.randint(5, 30)
                color = tuple(np.random.randint(50, 200, 3))
                draw.ellipse([x-r, y-r, x+r, y+r], fill=color, outline=color)
        
        # Add text representation
        try:
            font = ImageFont.load_default()
            # Wrap text
            words = prompt.split()
            lines = []
            current_line = []
            for word in words:
                current_line.append(word)
                if len(' '.join(current_line)) > 40:
                    current_line.pop()
                    lines.append(' '.join(current_line))
                    current_line = [word]
            if current_line:
                lines.append(' '.join(current_line))
            
            y_offset = size[1] - 30 * len(lines) - 10
            for line in lines:
                draw.text((10, y_offset), line[:50], fill=(255, 255, 255), font=font)
                y_offset += 30
        except:
            pass
        
        return img
    
    def create_height_map(self, image):
        """Create height/depth map from image"""
        # Convert to grayscale
        if len(np.array(image).shape) == 3:
            img_array = np.array(image.convert('L'))
        else:
            img_array = np.array(image)
        
        # Apply edge detection for structure
        edges = cv2.Canny(img_array, 50, 150)
        
        # Create depth from intensity
        depth = img_array.astype(np.float32) / 255.0
        
        # Add edge emphasis
        depth = depth + edges.astype(np.float32) * 0.3
        
        # Apply Gaussian blur for smoothness
        depth = cv2.GaussianBlur(depth, (15, 15), 0)
        
        # Normalize
        depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-6)
        
        return depth
    
    def create_mesh_from_depth(self, depth_map, image, resolution=100, smooth=True):
        """Create 3D mesh from depth map using marching cubes"""
        
        # Resample depth map to target resolution
        h, w = depth_map.shape
        resolution = min(resolution, 150)  # Limit for performance
        depth_resized = cv2.resize(depth_map, (resolution, resolution))
        
        # Create 3D volume with more interesting shapes
        volume = np.zeros((resolution, resolution, resolution))
        
        # Fill volume based on depth and gradient
        for i in range(resolution):
            for j in range(resolution):
                depth_value = depth_resized[i, j]
                # Create organic shapes using sine/cosine
                z_height = int(depth_value * (resolution - 1))
                
                # Add variation based on position
                variation = 0.3 * np.sin(i * np.pi / resolution) * np.cos(j * np.pi / resolution)
                adjusted_height = max(1, min(resolution-1, z_height + int(variation * resolution)))
                
                volume[i, j, :adjusted_height] = 1.0
                
                # Add surface texture
                if adjusted_height > 0:
                    texture_value = 0.5 + 0.5 * np.sin(i * 0.3) * np.cos(j * 0.3)
                    volume[i, j, adjusted_height-1] = texture_value
        
        # Apply smoothing if requested
        if smooth:
            volume = ndimage.gaussian_filter(volume, sigma=1.0)
        
        # Apply marching cubes to extract mesh
        try:
            verts, faces, normals, values = measure.marching_cubes(volume, level=0.5)
        except:
            verts, faces, normals, values = measure.marching_cubes(volume, level=0.3)
        
        # Normalize vertices to range [-1, 1]
        verts = verts / resolution
        verts = verts * 2 - 1  # Scale to [-1, 1]
        
        # Get colors from original image
        img_array = np.array(image.resize((resolution, resolution)))
        
        # Assign colors to vertices based on position
        vertex_colors = []
        for v in verts:
            x = int((v[0] + 1) / 2 * (resolution - 1))
            y = int((v[1] + 1) / 2 * (resolution - 1))
            x = max(0, min(resolution-1, x))
            y = max(0, min(resolution-1, y))
            
            if len(img_array.shape) == 3:
                color = img_array[y, x, :3] / 255.0
            else:
                color = [img_array[y, x] / 255.0] * 3
            vertex_colors.append(color)
        
        return verts, faces, np.array(vertex_colors)
    
    def save_as_obj(self, verts, faces, colors, filename):
        """Save mesh as OBJ file with colors"""
        with open(filename, 'w') as f:
            f.write("# NotY3dGenAI Generated Model\n")
            f.write("o generated_model\n")
            
            # Write vertices with colors
            for i, (v, c) in enumerate(zip(verts, colors)):
                f.write(f"v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f} {c[0]:.3f} {c[1]:.3f} {c[2]:.3f}\n")
            
            # Write faces
            for face in faces:
                f.write(f"f {face[0]+1} {face[1]+1} {face[2]+1}\n")
    
    def save_as_ply(self, verts, faces, colors, filename):
        """Save mesh as PLY file"""
        with open(filename, 'w') as f:
            f.write("ply\n")
            f.write("format ascii 1.0\n")
            f.write(f"element vertex {len(verts)}\n")
            f.write("property float x\n")
            f.write("property float y\n")
            f.write("property float z\n")
            f.write("property uchar red\n")
            f.write("property uchar green\n")
            f.write("property uchar blue\n")
            f.write(f"element face {len(faces)}\n")
            f.write("property list uchar int vertex_indices\n")
            f.write("end_header\n")
            
            # Write vertices
            for v, c in zip(verts, colors):
                f.write(f"{v[0]:.6f} {v[1]:.6f} {v[2]:.6f} {int(c[0]*255)} {int(c[1]*255)} {int(c[2]*255)}\n")
            
            # Write faces
            for face in faces:
                f.write(f"3 {face[0]} {face[1]} {face[2]}\n")
    
    def simplify_mesh(self, verts, faces, target_faces=10000):
        """Simplify mesh by reducing face count"""
        if len(faces) <= target_faces:
            return verts, faces
        
        # Intelligent simplification: keep important features
        step = max(1, len(faces) // target_faces)
        simplified_faces = faces[::step]
        
        # Get unique vertices
        used_vertices = set()
        for face in simplified_faces:
            used_vertices.update(face)
        
        # Remap vertex indices
        vertex_map = {old: new for new, old in enumerate(sorted(used_vertices))}
        new_verts = verts[list(used_vertices)]
        new_faces = np.array([[vertex_map[v] for v in face] for face in simplified_faces])
        
        return new_verts, new_faces
    
    def save_model(self, verts, faces, colors, prompt, format="obj"):
        """Save model in requested format"""
        timestamp = int(time.time())
        safe_prompt = "".join(c for c in prompt[:30] if c.isalnum() or c in (' ', '-', '_')).rstrip()
        filename = f"noty3d_{safe_prompt}_{timestamp}.{format}"
        
        # Save locally
        local_path = f"/content/noty3d_models/{filename}"
        
        if format == "obj":
            self.save_as_obj(verts, faces, colors, local_path)
        elif format == "ply":
            self.save_as_ply(verts, faces, colors, local_path)
        
        # Save to Drive
        drive_path = f"/content/drive/MyDrive/NotY3D_Models/{filename}"
        if format == "obj":
            self.save_as_obj(verts, faces, colors, drive_path)
        elif format == "ply":
            self.save_as_ply(verts, faces, colors, drive_path)
        
        # Save metadata
        metadata = {
            "prompt": prompt,
            "timestamp": timestamp,
            "format": format,
            "vertices": int(len(verts)),
            "faces": int(len(faces)),
            "device": str(self.device)
        }
        
        with open(f"/content/noty3d_models/{filename}.json", "w") as f:
            json.dump(metadata, f, indent=2)
        
        return local_path, drive_path
    
    def create_ui(self):
        """Create interactive UI"""
        
        # Style
        display(HTML("""
        <style>
            .noty3d-title {
                font-size: 42px;
                font-weight: bold;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                -webkit-background-clip: text;
                -webkit-text-fill-color: transparent;
                text-align: center;
                margin-bottom: 20px;
            }
            .info-box {
                background: linear-gradient(135deg, #667eea 20%, #764ba2 80%);
                color: white;
                padding: 15px;
                border-radius: 10px;
                margin: 10px 0;
                font-size: 14px;
            }
            .control-panel {
                background: #f8f9fa;
                padding: 20px;
                border-radius: 10px;
                margin: 10px 0;
            }
        </style>
        """))
        
        display(HTML('<div class="noty3d-title">🎨 NotY3dGenAI - 3D Model Generator</div>'))
        
        # Info box
        display(HTML(f"""
        <div class="info-box">
            ✅ <b>Device:</b> {self.device}<br>
            🚀 <b>Status:</b> Ready to generate 3D models<br>
            💾 <b>Storage:</b> Models auto-save to Google Drive/NotY3D_Models/
        </div>
        """))
        
        # Prompt input
        self.prompt_input = widgets.Textarea(
            value="A powerful muscular man with authoritative presence, standing confidently",
            placeholder="Describe the 3D model you want to generate...",
            description="🎯 Prompt:",
            layout=widgets.Layout(width='100%', height='120px')
        )
        
        # Quality controls
        self.resolution = widgets.IntSlider(
            value=100,
            min=60,
            max=150,
            step=10,
            description="🔍 Mesh Resolution:",
            style={'description_width': 'initial'}
        )
        
        self.quality = widgets.Dropdown(
            options=['High', 'Medium', 'Low'],
            value='Medium',
            description="✨ Quality:",
            style={'description_width': 'initial'}
        )
        
        self.smooth = widgets.Checkbox(
            value=True,
            description="Smooth Mesh",
            style={'description_width': 'initial'}
        )
        
        self.poly_count = widgets.IntSlider(
            value=10000,
            min=3000,
            max=20000,
            step=1000,
            description="🔺 Max Polygons:",
            style={'description_width': 'initial'}
        )
        
        self.output_format = widgets.Dropdown(
            options=['obj', 'ply'],
            value='obj',
            description="📁 Output Format:",
            style={'description_width': 'initial'}
        )
        
        # Generate button
        self.generate_btn = widgets.Button(
            description="🚀 Generate 3D Model",
            button_style='primary',
            layout=widgets.Layout(width='100%', height='50px')
        )
        
        self.status_output = widgets.Output()
        
        # Layout
        controls = widgets.VBox([
            self.prompt_input,
            widgets.HBox([self.resolution, self.quality]),
            widgets.HBox([self.smooth, self.poly_count]),
            self.output_format,
            self.generate_btn,
            self.status_output
        ], layout=widgets.Layout(padding='10px'))
        
        self.viewer_output = widgets.Output()
        
        # Main container
        container = widgets.HBox([
            widgets.VBox([controls], layout=widgets.Layout(width='40%')),
            widgets.VBox([
                widgets.HTML("<h3>📊 3D Viewer</h3>"),
                self.viewer_output
            ], layout=widgets.Layout(width='60%'))
        ])
        
        display(container)
        
        # Bind function
        self.generate_btn.on_click(self.generate_model)
    
    def generate_model(self, btn):
        """Generate 3D model"""
        with self.status_output:
            clear_output()
            start_time = time.time()
            
            print("🎨 Starting 3D model generation...")
            print(f"📝 Prompt: {self.prompt_input.value[:100]}...")
            print(f"⚙️ Settings: {self.quality.value}, Resolution: {self.resolution.value}")
            
            try:
                # Generate image
                print("📸 Generating image from prompt...")
                image = self.generate_fallback_image(self.prompt_input.value)
                
                # Display generated image
                print("✅ Image generated!")
                display(image)
                
                # Create depth map
                print("🗺️ Creating depth map...")
                depth_map = self.create_height_map(image)
                
                # Create mesh
                print("🔺 Generating 3D mesh...")
                resolution = self.resolution.value
                if self.quality.value == 'Low':
                    resolution = max(60, resolution // 2)
                elif self.quality.value == 'High':
                    resolution = min(150, resolution + 20)
                
                verts, faces, colors = self.create_mesh_from_depth(
                    depth_map, image, 
                    resolution=resolution,
                    smooth=self.smooth.value
                )
                
                print(f"📐 Initial mesh: {len(verts)} vertices, {len(faces)} faces")
                
                # Simplify if needed
                if len(faces) > self.poly_count.value:
                    print(f"📐 Simplifying mesh ({len(faces)} -> {self.poly_count.value} faces)...")
                    verts, faces = self.simplify_mesh(verts, faces, self.poly_count.value)
                
                # Save model
                print("💾 Saving model...")
                local_path, drive_path = self.save_model(
                    verts, faces, colors,
                    self.prompt_input.value,
                    self.output_format.value
                )
                
                # Time taken
                elapsed = time.time() - start_time
                
                print(f"\n✅ Model generated successfully!")
                print(f"⏱️ Time: {elapsed:.2f} seconds")
                print(f"🔺 Final vertices: {len(verts):,}")
                print(f"🔻 Final faces: {len(faces):,}")
                print(f"💾 Saved locally: {local_path}")
                print(f"☁️ Backed up to Drive: {drive_path}")
                
                # Display 3D model
                with self.viewer_output:
                    clear_output()
                    self.display_3d_model(verts, faces, colors)
                
                # Create download link
                from IPython.display import FileLink
                display(HTML(f"<br><h3>📥 Download your model:</h3>"))
                display(FileLink(local_path))
                
                print(f"\n💡 Model also available at: {drive_path}")
                
            except Exception as e:
                print(f"❌ Error: {str(e)}")
                import traceback
                traceback.print_exc()
                print("\n💡 Try reducing resolution or quality settings")
    
    def display_3d_model(self, verts, faces, colors):
        """Display 3D model using plotly"""
        try:
            # Sample for performance
            if len(verts) > 5000:
                step = len(verts) // 5000
                sampled_verts = verts[::step]
                sampled_colors = colors[::step]
                
                # Recreate faces with sampled vertices
                valid_faces = []
                for face in faces[:5000]:
                    if all(v % step == 0 for v in face):
                        valid_faces.append([v//step for v in face])
                display_faces = np.array(valid_faces[:3000])
                display_verts = sampled_verts
                display_colors = sampled_colors
            else:
                display_verts = verts
                display_faces = faces[:3000]
                display_colors = colors
            
            if len(display_faces) == 0:
                print("Not enough faces to display")
                return
            
            # Create mesh plot
            fig = go.Figure(data=[
                go.Mesh3d(
                    x=display_verts[:, 0],
                    y=display_verts[:, 1],
                    z=display_verts[:, 2],
                    i=display_faces[:, 0],
                    j=display_faces[:, 1],
                    k=display_faces[:, 2],
                    vertexcolor=display_colors,
                    intensity=display_verts[:, 2],
                    colorscale='Viridis',
                    opacity=0.85,
                    lighting=dict(ambient=0.6, diffuse=0.8, specular=0.3, roughness=0.5),
                    lightposition=dict(x=100, y=200, z=300)
                )
            ])
            
            fig.update_layout(
                scene=dict(
                    xaxis_title='X',
                    yaxis_title='Y',
                    zaxis_title='Z',
                    camera=dict(
                        eye=dict(x=1.5, y=1.5, z=1.5),
                        up=dict(x=0, y=1, z=0)
                    ),
                    aspectmode='data',
                    bgcolor='black'
                ),
                width=700,
                height=500,
                margin=dict(l=0, r=0, b=0, t=0),
                paper_bgcolor='black',
                plot_bgcolor='black'
            )
            
            fig.show()
            
        except Exception as e:
            print(f"⚠️ Could not display 3D model: {e}")

# Initialize app
print("""
╔══════════════════════════════════════════════════════════╗
║                                                          ║
║     🎨 NotY3dGenAI - Professional 3D Generator          ║
║                                                          ║
║     Features:                                            ║
║     • Text to 3D Model Generation                       ║
║     • Full Quality Controls                             ║
║     • Polygon & Resolution Settings                     ║
║     • Multiple Output Formats (OBJ/PLY)                 ║
║     • Auto-save to Google Drive                         ║
║     • Real-time 3D Viewer                               ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""")

# Create and launch app
app = NotY3dGenAI()

print("\n" + "="*60)
print("✨ NotY3dGenAI is ready!")
print("="*60)
print("\n💡 Tips for best results:")
print("   • Be descriptive in your prompts")
print("   • Use 'High' quality for detailed models")
print("   • Lower polygon count for faster loading")
print("   • Enable 'Smooth Mesh' for organic shapes")
print("   • Models are automatically saved to Google Drive")
print("\n🎯 Enter your prompt and click 'Generate 3D Model' to begin!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 Installing required packages...
✅ All packages installed successfully!

╔══════════════════════════════════════════════════════════╗
║                                                          ║
║     🎨 NotY3dGenAI - Professional 3D Generator          ║
║                                                          ║
║     Features:                                            ║
║     • Text to 3D Model Generation                       ║
║     • Full Quality Controls                             ║
║     • Polygon & Resolution Settings                     ║
║     • Multiple Output Formats (OBJ/PLY)                 ║
║     • Auto-save to Google Drive                         ║
║     • Real-time 3D Viewer                               ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝

✅ GPU detec


✨ NotY3dGenAI is ready!

💡 Tips for best results:
   • Be descriptive in your prompts
   • Use 'High' quality for detailed models
   • Lower polygon count for faster loading
   • Enable 'Smooth Mesh' for organic shapes
   • Models are automatically saved to Google Drive

🎯 Enter your prompt and click 'Generate 3D Model' to begin!
